# Within Reach? Mapping Park Accessibility in Manhattan

This project explores how accessible green space is across Manhattan by modeling a 5-minute walking radius around parks to approximate real-world access to nature in dense urban environments.

## Methodology

I used the NYC Parks spatial dataset and filtered it to Manhattan park geometries. Each park polygon was converted into a geospatial object, then buffered by approximately 400 meters to represent a 5-minute walk. The buffered areas were merged into a single accessibility layer and visualized alongside the original park boundaries in an interactive Folium map.

In [2]:
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import shape

# ---------------------------
# LOAD NYC PARKS DATA
# ---------------------------
parks = pd.read_json("https://data.cityofnewyork.us/resource/enfh-gkve.json")

# ---------------------------
# FILTER TO MANHATTAN
# ---------------------------
manhattan = parks[parks["borough"] == "M"].copy()
manhattan["acres"] = pd.to_numeric(manhattan["acres"], errors="coerce")

# ---------------------------
# BUILD GEOMETRY
# ---------------------------
geoms = []
names = []
acres = []

for _, row in manhattan.iterrows():
    try:
        geoms.append(shape(row["multipolygon"]))
        names.append(row.get("name311", "Park"))
        acres.append(row["acres"])
    except:
        pass

gdf = gpd.GeoDataFrame(
    {"name": names, "acres": acres, "geometry": geoms},
    crs="EPSG:4326"
)

# ---------------------------
# CREATE 5-MINUTE WALK BUFFERS
# ---------------------------
gdf_proj = gdf.to_crs(epsg=3857)
gdf_proj["buffer"] = gdf_proj.geometry.buffer(400)  # ~400 meters

coverage = gdf_proj["buffer"].union_all()
coverage = gpd.GeoSeries([coverage], crs="EPSG:3857").to_crs(epsg=4326)

# ---------------------------
# BUILD MAP
# ---------------------------
m = folium.Map(
    location=[40.7831, -73.9712],
    zoom_start=12,
    tiles="cartodbpositron",
    control_scale=True
)

# coverage layer
folium.GeoJson(
    coverage,
    style_function=lambda x: {
        "fillColor": "#9ecae1",
        "color": "#6baed6",
        "fillOpacity": 0.35,
        "weight": 1
    }
).add_to(m)

# park polygons with hover
for _, row in gdf.iterrows():
    try:
        folium.GeoJson(
            row["geometry"],
            style_function=lambda x: {
                "fillColor": "#2f4f4f",
                "color": "#2f4f4f",
                "weight": 0.8,
                "fillOpacity": 0.6
            },
            tooltip=folium.Tooltip(
                f"<b>{row['name']}</b><br>Acres: {row['acres']}"
            )
        ).add_to(m)
    except:
        pass

# title
title_html = """
<div style="
position: fixed;
top: 15px;
left: 50%;
transform: translateX(-50%);
z-index: 9999;
background-color: rgba(255,255,255,0.95);
padding: 10px 16px;
border-radius: 10px;
box-shadow: 0 1px 6px rgba(0,0,0,0.2);
font-family: Arial;
text-align: center;
">
<b>Within Reach? Park Access Across Manhattan</b><br>
<span style="font-size: 12px;">
Blue areas show ~5-minute walking access to green space
</span>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

# legend
legend_html = """
<div style="
position: fixed;
bottom: 25px;
left: 25px;
z-index: 9999;
background-color: rgba(255,255,255,0.92);
padding: 12px 14px;
border-radius: 8px;
box-shadow: 0 1px 6px rgba(0,0,0,0.2);
font-family: Arial;
font-size: 12px;
">
<b>Layers</b><br><br>
<span style="color:#6baed6;">●</span> Walkable park coverage<br>
<span style="color:#2f4f4f;">●</span> Parks (hover for details)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# insight box
insight_html = """
<div style="
position: fixed;
bottom: 120px;
left: 25px;
z-index: 9999;
background-color: rgba(255,255,255,0.92);
padding: 12px 14px;
border-radius: 8px;
box-shadow: 0 1px 6px rgba(0,0,0,0.2);
font-family: Arial;
font-size: 12px;
max-width: 260px;
">
<b>Key Insight</b><br><br>
Large parks create broad coverage zones, but access remains less immediate in some dense parts of Manhattan.
</div>
"""
m.get_root().html.add_child(folium.Element(insight_html))

m

## Key Findings

The map suggests that park access in Manhattan is shaped heavily by a few large anchor parks, especially Central Park, which generate broad walkable coverage zones. At the same time, smaller parks create more fragmented access patterns, meaning green space is present across the borough but not always evenly reachable in the same way. This makes Manhattan a useful case study in how dense urban form affects everyday access to nature.

## Limitations

This analysis uses a simplified 400-meter buffer as a proxy for a 5-minute walk and does not account for street networks, entrances, barriers, or park quality. As a result, the map estimates proximity to green space rather than exact pedestrian accessibility. Even so, it provides a useful high-level view of how park access is distributed across Manhattan.

## Next Steps

Future iterations could use street-network-based walking distances instead of simple spatial buffers, compare accessibility across boroughs, or incorporate population density to explore how park access varies relative to where people live.